# Calibrate H5 File Workflow

Run HDF5 calibration and inspect the fitted per-channel timing/lifetime outputs.


In [2]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from brighteyes_flim import calibrate_h5_file, show_h5_structure_html, sum_channel_applying_shifts
import brighteyes_flim.graph_tools as graph


In [3]:
Path.cwd().resolve()

PosixPath('/home/mdonato/myDev/BrightEyes-Flim-GIT/example notebooks')

## Calibration Parameters


In [ ]:
DATA_KEY = ('data', 'data_channels_extra')
INSPECT_DATA_KEY = DATA_KEY if isinstance(DATA_KEY, str) else DATA_KEY[0]

FILE_REFERENCE = '/mnt/DATA/Mixed Data/26-03-17_Convallaria_and_FLIMLABS_calibrated_Yellow_Slide/FLIMLABS_Yellow_slide_2_5ns-17-03-2026-16-18-22.h5'
FILE_DATA = '/mnt/DATA/Mixed Data/26-03-17_Convallaria_and_FLIMLABS_calibrated_Yellow_Slide/03_Convallaria_DFD_40MHz-17-03-2026-17-07-20_calib.h5'

TAU_REF = 2.5
REFERENCE_TYPE = 'ref'
FIT_MODE = 'model_shift'
FIT_TYPE = 'likelihood'
LASER_PERIOD_NS = None

CHANNEL_SKEW_TYPE = "phase_cross_correlation"
CHANNEL_SKEW_SOURCE = 'ref'
CHANNEL_SKEW_FIT_REFERENCE_CHANNEL = 12
CHANNEL_SKEW_FIT_UPSAMPLING = 10
CHANNEL_SKEW_FIT_APODIZE = False
OVERWRITE = True


## Run Calibration


In [ ]:
output_path = calibrate_h5_file(
    FILE_DATA,
    FILE_REFERENCE,
    data_key=DATA_KEY,
    reference_type=REFERENCE_TYPE,
    tau_ref=TAU_REF,
    fit_mode=FIT_MODE,
    fit_type=FIT_TYPE,
    channel_skew_type=CHANNEL_SKEW_TYPE,
    channel_skew_source=CHANNEL_SKEW_SOURCE,
    channel_skew_fit_reference_channel=CHANNEL_SKEW_FIT_REFERENCE_CHANNEL,
    channel_skew_fit_upsampling=CHANNEL_SKEW_FIT_UPSAMPLING,
    channel_skew_fit_apodize=CHANNEL_SKEW_FIT_APODIZE,
    period_ns=LASER_PERIOD_NS,
    overwrite=OVERWRITE,
)
print(output_path)


## Load Calibration Tables


In [ ]:
def get_calibration_group(hf, data_key=INSPECT_DATA_KEY, calibration_key="calibration"):
    calibration_root = hf[calibration_key]
    if data_key not in calibration_root or not isinstance(calibration_root[data_key], h5py.Group):
        raise KeyError(f"Expected group {calibration_key}/{data_key}; available: {list(calibration_root.keys())}")
    return calibration_root[data_key]


def calibration_summary_dataframe(group):
    return pd.DataFrame(
        {
            "channel": group["channel_index"][()],
            "channel_used_for_reference_in_time_skew": group["channel_used_for_reference_in_time_skew"][()],
            "fit_param_C": group["fit_param_C"][()],
            "fit_param_C_err": group["fit_param_C_err"][()],
            "tau_ns": group["tau_ns"][()],
            "tau_err_ns": group["tau_err_ns"][()],
            "tau_ref_ns": group["tau_ref_ns"][()],
            "common_delay_in_bins": group["common_delay_in_bins"][()],
            "common_delay_err_in_bins": group["common_delay_err_in_bins"][()],
            "common_delay_in_ns": group["common_delay_in_ns"][()],
            "common_delay_err_in_ns": group["common_delay_err_in_ns"][()],
            "channel_skew": group["channel_skew"][()],
            "channel_skew_est_err": group["channel_skew_est_err"][()],
            "fit_error": group["fit_error"][()],
        }
    )

with h5py.File(output_path, "r") as hf:
    calibration = get_calibration_group(hf)
    summary_df = calibration_summary_dataframe(calibration)
    attrs = dict(calibration.attrs)

summary_df


## Channel Fit Summary


In [ ]:
graph.plot_calibration_lifetime_summary(summary_df)


In [ ]:
data_keys = [DATA_KEY] if isinstance(DATA_KEY, str) else list(DATA_KEY)
summary_tables = []
labels = []
with h5py.File(output_path, "r") as hf:
    for data_key in data_keys:
        group = get_calibration_group(hf, data_key=data_key)
        summary_tables.append(calibration_summary_dataframe(group))
        source = group.attrs.get("channel_skew_source", "not stored")
        ref_key = group.attrs.get("channel_skew_fit_reference_data_key", data_key)
        ref_channel = group.attrs.get("channel_skew_fit_reference_channel_resolved", CHANNEL_SKEW_FIT_REFERENCE_CHANNEL)
        labels.append(f"{data_key} | {source} | ref {ref_key}:{ref_channel}")

graph.plot_calibration_shift_summary(
    summary_tables,
    labels=labels,
    reference_channel=CHANNEL_SKEW_FIT_REFERENCE_CHANNEL,
)


## Inspect One Channel Fit


In [ ]:
CHANNEL = 12

with h5py.File(output_path, "r") as hf:
    calibration = get_calibration_group(hf)
    calibrated_channels = calibration["channel_index"][()]
    column = int(np.where(calibrated_channels == CHANNEL)[0][0])
    laser_period_in_ns = float(calibration.attrs["laser_period_in_ns"])
    t_ns = np.arange(calibration["data_for_fit"].shape[0], dtype=float) * laser_period_in_ns / calibration["data_for_fit"].shape[0]

    traces = {
        "data_for_fit": calibration["data_for_fit"][:, column],
        "ref_for_fit": calibration["ref_for_fit"][:, column] if "ref_for_fit" in calibration else None,
        "ref_common_delay_realigned": calibration["ref_common_delay_realigned"][:, column] if "ref_common_delay_realigned" in calibration else None,
        "irf_for_fit": calibration["irf_for_fit"][:, column],
        "irf_common_delay_realigned": calibration["irf_common_delay_realigned"][:, column],
        "data_fitted": calibration["data_fitted"][:, column],
    }
    tau_ns = float(calibration["tau_ns"][column])
    delay_ns = float(calibration["common_delay_in_ns"][column])
    fit_error = float(calibration["fit_error"][column])

fig, ax = plt.subplots(figsize=(10, 4))
graph.plot_calibration_fit_traces(
    t_ns,
    traces,
    title=f"Channel {CHANNEL}: tau={tau_ns:.2f} ns, delay={delay_ns:.2f} ns, RMSE={fit_error:.3g}",
    ax=ax,
)


## HDF5 Structure


In [ ]:
show_h5_structure_html(output_path)


In [ ]:
output_path
